# PROYECTO FINAL: APLICACIONES ANALÍTICAS DE BIG DATA (UAPA)
## Auditoría de Experiencia y Sentimiento de Marca en Telecomunicaciones vía YouTube Data API v3
### Caso de Estudio: Claro República Dominicana

**Integrantes del Equipo:**
- **Audric Rosario** (*Lead Data Engineering & NLP Modeling*)
- **Orlando Benítez** (*Lead Business Intelligence & Executive Strategy*)

---
### Objetivos del Notebook:
1. Demostrar la extracción ética y prudente de comentarios reales usando **YouTube Data API v3**.
2. Ejecutar el preprocesamiento lingüístico del español dominicano.
3. Implementar un pipeline de **NLP con arquitectura Transformer preentrenada** para clasificación de sentimiento y detección de tópicos.
4. Calcular el **Net Sentiment Score (NSS)** e indicadores de gestión empresarial.

### 1. Carga de Librerías y Configuración del Entorno

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Añadir src al path para reutilizar módulos modulares
sys.path.append(os.path.abspath('../src'))
from preprocesamiento import preprocesar_dataframe, limpiar_texto
from modelado_nlp import SentimentTransformerPipeline

print("Entorno inicializado correctamente.")

### 2. Carga y Exploración de Datos Crudos de YouTube (Claro RD)

In [ ]:
raw_path = '../data/raw/youtube_claro_raw.csv'
df_raw = pd.read_csv(raw_path)
print(f"Total de registros cargados: {len(df_raw)}")
df_raw[['video_title', 'author', 'comment_text', 'published_at', 'like_count']].head()

### 3. Pipeline de Preprocesamiento y Limpieza de Texto en Español

In [ ]:
df_clean = preprocesar_dataframe(df_raw, columna_texto='comment_text')
print("Muestra de texto original vs texto limpio:")
df_clean[['comment_text', 'clean_text', 'topic_category']].head(5)

### 4. Clasificación de Sentimiento con Modelo Transformer Preentrenado

In [ ]:
pipeline_sentimiento = SentimentTransformerPipeline()
df_scored = pipeline_sentimiento.procesar_dataframe(df_clean, columna_texto='clean_text')
df_scored[['clean_text', 'topic_category', 'sentiment_label', 'sentiment_score']].head(8)

### 5. Cálculo del Net Sentiment Score (NSS) y Métricas Ejecutivas

In [ ]:
total = len(df_scored)
dist_sent = df_scored['sentiment_label'].value_counts()
pct_pos = (dist_sent.get('POSITIVO', 0) / total) * 100
pct_neg = (dist_sent.get('NEGATIVO', 0) / total) * 100
pct_neu = (dist_sent.get('NEUTRO', 0) / total) * 100
nss = pct_pos - pct_neg

print(f"--- MÉTRICAS GERENCIALES CLARO DOMINICANA ---")
print(f"Volumen total: {total} interacciones")
print(f"Positivos: {pct_pos:.2f}%")
print(f"Negativos: {pct_neg:.2f}%")
print(f"Neutros:   {pct_neu:.2f}%")
print(f"Net Sentiment Score (NSS): {nss:+.2f}%")

### 6. Visualización Interactiva con Plotly

In [ ]:
fig_pie = px.pie(
    df_scored,
    names='sentiment_label',
    color='sentiment_label',
    color_discrete_map={'POSITIVO': '#2ECC71', 'NEGATIVO': '#E74C3C', 'NEUTRO': '#95A5A6'},
    hole=0.5,
    title="Distribución de Sentimiento Global de Claro Dominicana en YouTube"
)
fig_pie.show()

In [ ]:
cat_summary = df_scored.groupby(['topic_category', 'sentiment_label']).size().reset_index(name='conteo')
fig_bar = px.bar(
    cat_summary,
    x='topic_category',
    y='conteo',
    color='sentiment_label',
    barmode='group',
    color_discrete_map={'POSITIVO': '#2ECC71', 'NEGATIVO': '#E74C3C', 'NEUTRO': '#95A5A6'},
    title="Sentimiento de Marca por Categoría de Servicio (Claro RD)",
    labels={'topic_category': 'Servicio', 'conteo': 'Volumen de Comentarios'}
)
fig_bar.show()

### 7. Exportación del Dataset Procesado
Se guarda el dataset enriquecido para consumo directo del Dashboard Ejecutivo.

In [ ]:
out_path = '../data/processed/youtube_claro_processed.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_scored.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Dataset procesado guardado exitosamente en: {out_path}")